In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Surgical Patch (Option B, PR #548)
# Patches the installed AR casanovo classes in-process at class level.
# Must run FIRST. No kernel restart needed — patches persist in memory.
#
# What the AR install has:          What we patch it to:
#   PeptideDecoder  → no embed()      → embed() with all-False tgt_mask
#   Spec2Pep        → GT tokens       → zero tokens in _forward_step
#   Spec2Pep.forward→ beam_search     → calls _forward_step
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings, inspect
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'casanovo>=5.0.0', 'pyteomics', 'lxml', 'remotezip', 'appdirs'], check=True)

import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep

# ── Capture the AR (inherited) embed BEFORE we overwrite it ──────────
# In AR casanovo, PeptideDecoder has no embed() → it inherits from
# AnalyteTransformerDecoder. Capturing here gives us the parent's embed
# to call inside our NAR version (equivalent to super().embed(…)).
_ar_embed_original = PeptideDecoder.embed

# ─────────────────────────────────────────────────────────────────────
# PATCH 1 — PeptideDecoder.embed
# Source: PR #548 transformers.py
# Replaces the causal upper-triangular tgt_mask the parent builds with
# an all-False (L×L) mask, allowing every position to attend to every
# other position — the core of non-autoregressive parallel decoding.
# ─────────────────────────────────────────────────────────────────────
def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               **kwargs):
    if tokens is None:
        # Match base-class convention: empty (1, 0) tensor on correct device
        tokens = torch.tensor([[]], dtype=torch.float,
                               device=next(self.parameters()).device)
    L = tokens.shape[1] + 1          # +1 for the prepended global token
    tgt_mask = torch.zeros((L, L), dtype=torch.bool, device=tokens.device)
    return _ar_embed_original(
        self, tokens, *args,
        memory=memory,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
        tgt_mask=tgt_mask,
        **kwargs,
    )

PeptideDecoder.embed = _nar_embed

# ─────────────────────────────────────────────────────────────────────
# PATCH 2 — Spec2Pep._forward_step
# Source: PR #548 model.py
# AR version feeds ground-truth tokens (teacher forcing).
# NAR version feeds an all-zero tensor so the decoder predicts ALL
# positions in a single parallel pass, with no access to prior outputs.
# ─────────────────────────────────────────────────────────────────────
def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    # Explicit .to(dev) makes direct profiling calls safe (Lightning
    # normally handles this, but we call the model outside a Trainer).
    mzs       = mzs.to(dev)
    ints      = ints.to(dev)
    precursors = precursors.to(dev)

    memories, mem_masks = self.encoder(mzs, ints)

    if seqs is not None:                    # training: match GT length
        zero_tokens = torch.zeros_like(seqs.to(dev))
    else:                                   # inference: full max length
        zero_tokens = torch.zeros(
            (mzs.shape[0], self.max_peptide_len),
            dtype=torch.long, device=dev,
        )
    scores = self.decoder(
        tokens=zero_tokens,
        memory=memories,
        memory_key_padding_mask=mem_masks,
        precursors=precursors,
    )
    return scores, seqs

Spec2Pep._forward_step = _nar_forward_step

# ─────────────────────────────────────────────────────────────────────
# PATCH 3 — Spec2Pep.forward
# Source: PR #548 model.py
# AR version called beam_search_decode; NAR delegates to _forward_step.
# ─────────────────────────────────────────────────────────────────────
def _nar_forward(self, batch):
    return self._forward_step(batch)

Spec2Pep.forward = _nar_forward

# ── Verification ──────────────────────────────────────────────────────
try:
    _ok_embed = 'tgt_mask = torch.zeros' in inspect.getsource(_nar_embed)
    _ok_fwd   = 'zero_tokens' in inspect.getsource(_nar_forward_step)
except (OSError, TypeError):
    # Fallback: identity check (works even when source is unavailable)
    _ok_embed = True
    _ok_fwd   = True

_ok_id_embed = (PeptideDecoder.embed  is _nar_embed)
_ok_id_fwd   = (Spec2Pep._forward_step is _nar_forward_step)
_ok_id_fwd2  = (Spec2Pep.forward       is _nar_forward)

print('\n── NAR Patch Status ──────────────────────────────────────────')
print(f'  PeptideDecoder.embed  patched (identity)  : {"✓" if _ok_id_embed else "✗ FAILED"}')
print(f'  Spec2Pep._forward_step patched (identity) : {"✓" if _ok_id_fwd   else "✗ FAILED"}')
print(f'  Spec2Pep.forward       patched (identity) : {"✓" if _ok_id_fwd2  else "✗ FAILED"}')
print(f'  all-False tgt_mask in embed source        : {"✓" if _ok_embed    else "~ (source check skipped)"}')
print(f'  zero_tokens in _forward_step source       : {"✓" if _ok_fwd      else "~ (source check skipped)"}')
print('──────────────────────────────────────────────────────────────')

if not (_ok_id_embed and _ok_id_fwd and _ok_id_fwd2):
    raise RuntimeError('One or more NAR patches failed — check errors above.')

print('\nNAR patches applied ✓  No kernel restart needed.')
print('Continue → run Cell 1 (Setup) next.')


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip



── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed  patched (identity)  : ✓
  Spec2Pep._forward_step patched (identity) : ✓
  Spec2Pep.forward       patched (identity) : ✓
  all-False tgt_mask in embed source        : ✓
  zero_tokens in _forward_step source       : ✓
──────────────────────────────────────────────────────────────

NAR patches applied ✓  No kernel restart needed.
Continue → run Cell 1 (Setup) next.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from pathlib import Path
from tqdm import tqdm

torch.manual_seed(42)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0

N_PEAKS = 150     # config default max_peaks
AVG_PEP = 12      # median observed peptide length (for synthetic benchmark)

# ── NAR profiling constants ─────────────────────────────────────────
N_SUBSET         = 6000   # spectra in MGF subset (buffer ensures ≥5000 pass charge filter)
N_TIMING_SPECTRA = 5000   # spectra to time per batch size
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 10     # warm-up batches before timing each batch size

# torch.profiler settings (batch size 1 only)
PROF_WARMUP = 20
PROF_ACTIVE = 50

def _sync():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

print(f'Device : {DEVICE} | GPU: {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'N_SUBSET={N_SUBSET} | N_TIMING_SPECTRA={N_TIMING_SPECTRA}')
print(f'Batch sizes : {BATCH_SIZES}')
print(f'Profiler    : warmup={PROF_WARMUP} active={PROF_ACTIVE} (bs=1 only)')
os.makedirs('results', exist_ok=True)

Device : cuda | GPU: NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8
N_SUBSET=6000 | N_TIMING_SPECTRA=5000
Batch sizes : [1, 8, 32, 128, 512]
Profiler    : warmup=20 active=50 (bs=1 only)


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Download MGF via HTTP range request (no full zip)
# ═══════════════════════════════════════════════════════════════════
from remotezip import RemoteZip
import shutil
 
ZIP_URL  = 'https://zenodo.org/records/12587317/files/mgf_data.zip?download=1'
TARGET   = 'multi-enzyme-simple.test.mgf'
MGF_PATH = TARGET
 
if not (os.path.exists(MGF_PATH) and os.path.getsize(MGF_PATH) > 1e6):
    with RemoteZip(ZIP_URL) as zf:
        src = next(n for n in zf.namelist() if TARGET in n)
        zf.extract(src, '.')
    if src != MGF_PATH and os.path.exists(src):
        shutil.move(src, MGF_PATH)
        top = src.split('/')[0]
        if os.path.isdir(top): shutil.rmtree(top, ignore_errors=True)
 
print(f'{MGF_PATH}  ({os.path.getsize(MGF_PATH)/1e6:.1f} MB)')

multi-enzyme-simple.test.mgf  (300.9 MB)


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — EDA: parse MGF + plots
# ═══════════════════════════════════════════════════════════════════
import re as _re
def parse_mgf(path):
    records, spec, peaks, in_s = [], {}, [], False
    with open(path, 'r', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.upper() == 'BEGIN IONS':
                spec, peaks, in_s = {}, [], True
            elif line.upper() == 'END IONS':
                if in_s:
                    records.append({'pepmass': spec.get('_pm', 0.0),
                                    'charge':  spec.get('_ch', 1),
                                    'n_peaks': len(peaks)})
                in_s = False
            elif in_s:
                if '=' in line:
                    k, _, v = line.partition('='); k = k.strip().upper()
                    if k == 'PEPMASS': spec['_pm'] = float(v.strip().split()[0])
                    elif k == 'CHARGE': spec['_ch'] = int(_re.sub(r'[^\d]', '', v.strip()) or '1')
                else:
                    p = line.split()
                    if p:
                        try: peaks.append(float(p[0]))
                        except ValueError: pass
    return pd.DataFrame(records)
 
eda = parse_mgf(MGF_PATH)
print(f'Spectra : {len(eda):,}')
print(f'Charge  : +{eda.charge.min()} to +{eda.charge.max()} | '
      f'+2: {int((eda.charge==2).sum()):,}  +3: {int((eda.charge==3).sum()):,}')
print(f'm/z     : {eda.pepmass.min():.1f} – {eda.pepmass.max():.1f}')
print(f'Peaks   : {eda.n_peaks.mean():.0f} avg  (min {eda.n_peaks.min()}, max {eda.n_peaks.max()})')
 
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('EDA — multi-enzyme-simple.test.mgf', fontweight='bold')
vc = eda.charge.value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white')
axes[0].set(title='Charge distribution', xlabel='Charge', ylabel='Count')
for bar, v in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vc.values)*0.01,
                 f'{v:,}', ha='center', fontsize=7)
axes[1].hist(eda.pepmass.clip(upper=3000), bins=60, color='darkorange', edgecolor='white')
axes[1].set(title='Precursor m/z (clip @3000)', xlabel='m/z', ylabel='Count')
axes[2].hist(eda.n_peaks.clip(upper=500), bins=60, color='seagreen', edgecolor='white')
axes[2].set(title='Peaks/spectrum (clip @500)', xlabel='Peaks', ylabel='Count')
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: results/eda_plots.png')

Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Saved: results/eda_plots.png


In [6]:
 
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Loading Casanovo model via Python API
# Uses the exact same checkpoint-finding function the CLI uses.
# ModelRunner handles all Lightning version compatibility internally.
# ═══════════════════════════════════════════════════════════════════
import appdirs
from casanovo.casanovo import _get_model_weights
from casanovo.denovo.model_runner import ModelRunner
from casanovo.config import Config
 
cache_dir = Path(appdirs.user_cache_dir("casanovo", False, opinion=False))
print(f'Cache dir : {cache_dir}')
ckpt_path  = _get_model_weights(cache_dir)   # finds cached or downloads
print(f'Checkpoint: {ckpt_path}  ({os.path.getsize(str(ckpt_path))/1e6:.0f} MB)')
 
config = Config()
runner = ModelRunner(config=config, model_filename=str(ckpt_path))
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model  = runner.model.to(DEVICE).eval()
 
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel class : {type(model).__name__}')
print(f'Parameters  : {n_params/1e6:.1f}M')
print(f'dim_model   : {model.encoder.latent_spectrum.shape[-1]}')
print(f'Enc layers  : {model.encoder.transformer_encoder.num_layers}')
try:
    print(f'Dec layers  : {model.decoder.transformer_decoder.num_layers}')
except AttributeError:
    pass
print(f'n_beams     : {model.n_beams} | max_peptide_len: {model.max_peptide_len}')
# ── Verify NAR patches are live on the loaded model ─────────────────
print('\n── NAR verification on loaded model ──')
import inspect as _insp
try:
    _v_embed = 'tgt_mask = torch.zeros' in _insp.getsource(model.decoder.__class__.embed)
    _v_fwd   = 'zero_tokens'            in _insp.getsource(model._forward_step)
except (OSError, TypeError):
    _v_embed = (model.decoder.__class__.embed    is _nar_embed)
    _v_fwd   = (model.__class__._forward_step    is _nar_forward_step)

print(f'  Decoder embed → full attention (NAR) : {"✓" if _v_embed else "✗  — re-run Cell 0!"}')
print(f'  _forward_step → zero tokens (NAR)    : {"✓" if _v_fwd   else "✗  — re-run Cell 0!"}')
print(f'  beam_search_decode present (unused)  : {hasattr(model, "beam_search_decode")}')
if not (_v_embed and _v_fwd):
    raise RuntimeError('NAR patches not active on loaded model. Run Cell 0 first.')
print(f'  max_peptide_len = {model.max_peptide_len}')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: [+25.980265]-, [Acetyl]-, C[Carbamidomethyl], M[Oxidation], [Ammonia-loss]-, Q[Deamidated], [Carbamyl]-, N[Deamidated]


Cache dir : /home/zeus/.cache/casanovo
Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt  (575 MB)

Model class : Spec2Pep
Parameters  : 47.9M
dim_model   : 512
Enc layers  : 9
Dec layers  : 9
n_beams     : 1 | max_peptide_len: 100

── NAR verification on loaded model ──
  Decoder embed → full attention (NAR) : ✓
  _forward_step → zero tokens (NAR)    : ✓
  beam_search_decode present (unused)  : True
  max_peptide_len = 100


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Build 6 000-spectrum MGF subset + Lance DataModule
# Auto-rebuilds the Lance if N_SUBSET or max_charge changed from the
# previous run (e.g. old subset had only 100 spectra).
# ═══════════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule
import shutil as _shutil

SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = os.path.join(os.getcwd(), 'lance_cache')
os.makedirs(LANCE_DIR, exist_ok=True)

MODEL_MAX_CHARGE = model.decoder.charge_encoder.num_embeddings
print(f'Model max_charge: {MODEL_MAX_CHARGE}')

# ── Helpers ──────────────────────────────────────────────────────────
def _count_mgf_spectra(path):
    if not os.path.exists(path):
        return 0
    with open(path, 'r', errors='replace') as f:
        return f.read().count('BEGIN IONS')

def write_subset_mgf(src, dest, n):
    count, buf, in_s = 0, [], False
    with open(src, 'r', errors='replace') as fin, open(dest, 'w') as fout:
        for line in fin:
            if count >= n:
                break
            if line.strip().upper() == 'BEGIN IONS':
                in_s = True; buf = [line]
            elif line.strip().upper() == 'END IONS':
                buf.append(line); fout.writelines(buf)
                count += 1; in_s = False; buf = []
            elif in_s:
                buf.append(line)
    return count

# ── Rebuild MGF subset if it is missing or too small ────────────────
_existing_n = _count_mgf_spectra(SUBSET_MGF)
if _existing_n < N_SUBSET:
    if _existing_n > 0:
        print(f'Old SUBSET_MGF has only {_existing_n} spectra (need {N_SUBSET}). Rebuilding…')
        os.remove(SUBSET_MGF)
    wrote = write_subset_mgf(MGF_PATH, SUBSET_MGF, N_SUBSET)
    print(f'Created: {SUBSET_MGF}  ({wrote} spectra)')
else:
    print(f'Reusing: {SUBSET_MGF}  ({_existing_n} spectra)')

# ── Rebuild Lance if N_SUBSET or max_charge changed ──────────────────
# Cache key encodes both so any change triggers a clean rebuild.
_mc_marker  = os.path.join(LANCE_DIR, '.cache_key')
_lance_test = os.path.join(LANCE_DIR, 'test.lance')
_cache_key  = f'{MODEL_MAX_CHARGE}_{N_SUBSET}'
_prev_key   = open(_mc_marker).read().strip() if os.path.exists(_mc_marker) else 'none'

if _prev_key != _cache_key:
    if os.path.exists(_lance_test):
        _shutil.rmtree(_lance_test)
        print(f'Deleted stale Lance (was {_prev_key!r}, now {_cache_key!r})')
    with open(_mc_marker, 'w') as f:
        f.write(_cache_key)
    print(f'Building Lance  cache_key={_cache_key} …')
else:
    print(f'Reusing Lance   cache_key={_cache_key} ✓')

# ── DataModule (bs=1 for setup / sanity check) ───────────────────────
dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
dm.setup(stage='test', annotated=False)
print('DataModule (bs=1) ready.')

# ── Sanity-check first batch ─────────────────────────────────────────
_it   = iter(dm.predict_dataloader())
_b    = next(_it); del _it
_mzs, _ints, _precs, _ = model._process_batch(_b)
_charge = _precs[0, 1].item() if _precs.ndim == 2 else _precs[1].item()
assert _charge <= MODEL_MAX_CHARGE, \
    f'Charge {_charge} > model max_charge {MODEL_MAX_CHARGE}'
print(f'First batch  mzs={_mzs.shape}  precs={_precs.shape}')
print(f'Precursor [0]: mass={_precs[0,0]:.1f}  charge={_charge:.0f}  mz={_precs[0,2]:.1f} ✓')
print(f'Subset ready: {N_SUBSET} spectra  (timing target: {N_TIMING_SPECTRA})')

Model max_charge: 4
Created: subset_profile.mgf  (6000 spectra)
Building Lance  cache_key=4_6000 …


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

DataModule (bs=1) ready.
First batch  mzs=torch.Size([1, 42])  precs=torch.Size([1, 3])
Precursor [0]: mass=3370.5  charge=3  mz=1124.5 ✓
Subset ready: 6000 spectra  (timing target: 5000)


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — NAR Stage Timing: 5 batch sizes × 5 000 spectra each
# ═══════════════════════════════════════════════════════════════════════
import threading

AR_REAL_ENC_MS   = 8.15
AR_REAL_DEC_MS   = 367.15
AR_REAL_TOTAL_MS = 377.46
AR_REAL_TP       = 2.6

# ── GPU monitor ──────────────────────────────────────────────────────
_gpu_samples = []
_stop_gpu    = threading.Event()

def _gpu_monitor_fn():
    import subprocess as _sp
    while not _stop_gpu.is_set():
        r = _sp.run(['nvidia-smi',
                     '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True)
        if r.returncode == 0:
            try:
                u, m = r.stdout.strip().split(', ')
                _gpu_samples.append((int(u), float(m) / 1024))
            except Exception:
                pass
        time.sleep(0.5)

_gpu_thread = threading.Thread(target=_gpu_monitor_fn, daemon=True)
_gpu_thread.start()

# ── Shared encoder-hook state ─────────────────────────────────────────
_enc_hook_buf = []

def _enc_pre_hook(module, inp):
    _sync()
    module._t0_hook = time.perf_counter()

def _enc_post_hook(module, inp, out):
    _sync()
    _enc_hook_buf.append((time.perf_counter() - module._t0_hook) * 1000)

# ═════════════════════════════════════════════════════════════════════
timing_summary = {}

for bs in BATCH_SIZES:
    print(f'\n══ Timing NAR  batch_size={bs:4d} ══')

    _dm_bs = DeNovoDataModule(
        lance_dir=LANCE_DIR,
        test_paths=[SUBSET_MGF],
        eval_batch_size=bs,
        tokenizer=runner.model.tokenizer,
        max_charge=MODEL_MAX_CHARGE,
        n_workers=0,
    )
    _dm_bs.setup(stage='test', annotated=False)

    _hp = model.encoder.register_forward_pre_hook(_enc_pre_hook)
    _hq = model.encoder.register_forward_hook(_enc_post_hook)

    # ── Warm-up ───────────────────────────────────────────────────────
    _w_iter = iter(_dm_bs.predict_dataloader())
    with torch.no_grad():
        for _w in range(N_WARMUP_BATCHES):
            try:
                _wb = next(_w_iter)
            except StopIteration:
                break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm = _wm.to(DEVICE); _wi = _wi.to(DEVICE); _wp = _wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            _wz = torch.zeros((_wm.shape[0], model.max_peptide_len),
                               dtype=torch.long, device=DEVICE)
            model.decoder(tokens=_wz, memory=_wme,
                          memory_key_padding_mask=_wmk, precursors=_wp)
    del _w_iter
    _sync()

    # ── Timing storage ────────────────────────────────────────────────
    _t = {k: [] for k in ['fetch', 'h2d', 'enc', 'nar', 'write', 'total', 'tp']}
    _loader = _dm_bs.predict_dataloader()
    _it     = iter(_loader)
    n_spec  = 0
    pbar    = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')

    while n_spec < N_TIMING_SPECTRA:
        _enc_hook_buf.clear()

        # Fetch
        _sync(); t0 = time.perf_counter()
        try:
            batch = next(_it)
        except StopIteration:
            _it = iter(_loader)
            batch = next(_it)
        t_fetch = (time.perf_counter() - t0) * 1000

        # H2D + process_batch
        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs   = mzs.to(DEVICE)
        ints  = ints.to(DEVICE)
        precs = precs.to(DEVICE)
        _sync(); t_h2d = (time.perf_counter() - t0) * 1000
        actual_bs = mzs.shape[0]

        with torch.no_grad():
            # Encoder (hook captures its own time independently)
            memories, mem_masks = model.encoder(mzs, ints)
            t_enc = _enc_hook_buf[0] if _enc_hook_buf else 0.0

            # NAR Decoder — single parallel pass over all positions
            zero_tokens = torch.zeros(
                (actual_bs, model.max_peptide_len),
                dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = model.decoder(
                tokens=zero_tokens,
                memory=memories,
                memory_key_padding_mask=mem_masks,
                precursors=precs,
            )
            _sync(); t_nar = (time.perf_counter() - t0) * 1000

        # Write
        t0 = time.perf_counter()
        pred_tok = scores.argmax(dim=-1).cpu()
        _out = [{'tokens': tok.tolist()} for tok in pred_tok]
        t_write = (time.perf_counter() - t0) * 1000

        t_total = t_fetch + t_h2d + t_enc + t_nar + t_write

        _t['fetch'].append(t_fetch  / actual_bs)
        _t['h2d'].append(t_h2d     / actual_bs)
        _t['enc'].append(t_enc     / actual_bs)
        _t['nar'].append(t_nar     / actual_bs)
        _t['write'].append(t_write / actual_bs)
        _t['total'].append(t_total / actual_bs)
        _t['tp'].append(actual_bs  / (t_total / 1000))

        n_spec += actual_bs
        pbar.update(actual_bs)
        if n_spec >= N_TIMING_SPECTRA:
            break

    pbar.close()
    _hp.remove(); _hq.remove()

    def _p(a, q): return np.percentile(a, q)
    timing_summary[bs] = {
        'n_spec'     : n_spec,
        'n_batches'  : len(_t['total']),
        'fetch_mean' : np.mean(_t['fetch']),
        'h2d_mean'   : np.mean(_t['h2d']),
        'enc_mean'   : np.mean(_t['enc']),
        'nar_mean'   : np.mean(_t['nar']),
        'write_mean' : np.mean(_t['write']),
        'total_mean' : np.mean(_t['total']),
        'total_p50'  : _p(_t['total'], 50),
        'total_p95'  : _p(_t['total'], 95),
        'throughput' : np.mean(_t['tp']),
        'raw'        : _t,
    }
    s = timing_summary[bs]
    print(f'  spectra={n_spec}  batches={len(_t["total"])}')
    print(f'  total : {s["total_mean"]:7.2f} ms/spec  '
          f'p50={s["total_p50"]:.2f}  p95={s["total_p95"]:.2f}')
    print(f'  enc   : {s["enc_mean"]:7.2f} ms/spec  '
          f'nar={s["nar_mean"]:.2f} ms/spec')
    print(f'  tp    : {s["throughput"]:.1f} spec/s  '
          f'(AR baseline: {AR_REAL_TP} spec/s)')

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

_stop_gpu.set()
time.sleep(1.0)

gpu_util_mean = np.mean([s[0] for s in _gpu_samples]) if _gpu_samples else 0
gpu_vram_peak = np.max([s[1] for s in _gpu_samples]) if _gpu_samples else 0

# ── Stage breakdown table (bs=1) ─────────────────────────────────────
s1 = timing_summary[1]
r1 = s1['raw']
df_stage = pd.DataFrame([
    {'Stage': 'DataLoader fetch',    'mean_ms': s1['fetch_mean'],
     'p50_ms': np.percentile(r1['fetch'], 50), 'p95_ms': np.percentile(r1['fetch'], 95)},
    {'Stage': 'H2D transfer',        'mean_ms': s1['h2d_mean'],
     'p50_ms': np.percentile(r1['h2d'],   50), 'p95_ms': np.percentile(r1['h2d'],   95)},
    {'Stage': 'SpectrumEncoder',     'mean_ms': s1['enc_mean'],
     'p50_ms': np.percentile(r1['enc'],   50), 'p95_ms': np.percentile(r1['enc'],   95)},
    {'Stage': 'NAR Decoder (1 pass)','mean_ms': s1['nar_mean'],
     'p50_ms': np.percentile(r1['nar'],   50), 'p95_ms': np.percentile(r1['nar'],   95)},
    {'Stage': 'Output write',        'mean_ms': s1['write_mean'],
     'p50_ms': np.percentile(r1['write'], 50), 'p95_ms': np.percentile(r1['write'], 95)},
    {'Stage': 'TOTAL per spectrum',  'mean_ms': s1['total_mean'],
     'p50_ms': s1['total_p50'],                'p95_ms': s1['total_p95']},
]).round(3)

df_throughput = pd.DataFrame([{
    'batch_size'        : bs,
    'total_ms_per_spec' : timing_summary[bs]['total_mean'],
    'throughput_spec_s' : timing_summary[bs]['throughput'],
    'enc_ms_per_spec'   : timing_summary[bs]['enc_mean'],
    'nar_ms_per_spec'   : timing_summary[bs]['nar_mean'],
    'p50_ms'            : timing_summary[bs]['total_p50'],
    'p95_ms'            : timing_summary[bs]['total_p95'],
} for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (NAR, bs=1, {s1["n_spec"]} spectra) ──')
print(df_stage.to_string(index=False))
print(f'\n── Multi-batch throughput ──')
print(df_throughput.to_string(index=False))
print(f'\nGPU util (mean): {gpu_util_mean:.0f}%  |  Peak VRAM: {gpu_vram_peak:.2f} GB')
print(f'AR baseline (bs=1): {AR_REAL_TOTAL_MS:.1f} ms/spec  |  {AR_REAL_TP} spec/s')

# ── FIX: pre-compute target strings to avoid backslash-in-f-string
#    (Python < 3.12 forbids backslash escapes inside f-string expressions)
_total_ms_bs1 = s1['total_mean']
_msg_35 = 'MEETS ✓' if _total_ms_bs1 <= 35 else f'FAILS — {_total_ms_bs1:.1f} ms'
_msg_50 = 'MEETS ✓' if _total_ms_bs1 <= 50 else f'FAILS — {_total_ms_bs1:.1f} ms'
print(f'35 ms target (bs=1): {_msg_35}')
print(f'50 ms target (bs=1): {_msg_50}')

df_stage.to_csv('results/nar_stage_timing_bs1.csv',     index=False)
df_throughput.to_csv('results/nar_throughput_all_bs.csv', index=False)
print('\nSaved: nar_stage_timing_bs1.csv | nar_throughput_all_bs.csv')


══ Timing NAR  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:53<00:00, 44.21spec/s]


  spectra=5000  batches=5000
  total :   22.20 ms/spec  p50=21.31  p95=29.93
  enc   :    7.89 ms/spec  nar=12.67 ms/spec
  tp    : 45.6 spec/s  (AR baseline: 2.6 spec/s)

══ Timing NAR  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:10, 497.84spec/s]                        

  spectra=5120  batches=40
  total :    2.00 ms/spec  p50=1.99  p95=2.09
  enc   :    0.55 ms/spec  nar=1.24 ms/spec
  tp    : 501.2 spec/s  (AR baseline: 2.6 spec/s)

══ Timing NAR  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:11, 463.51spec/s]                        


  spectra=5120  batches=10
  total :    2.15 ms/spec  p50=2.15  p95=2.20
  enc   :    0.66 ms/spec  nar=1.30 ms/spec
  tp    : 464.3 spec/s  (AR baseline: 2.6 spec/s)

── Stage breakdown (NAR, bs=1, 5000 spectra) ──
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.334   1.276   1.845
        H2D transfer    0.210   0.197   0.295
     SpectrumEncoder    7.887   7.480  11.657
NAR Decoder (1 pass)   12.669  12.174  16.541
        Output write    0.103   0.094   0.132
  TOTAL per spectrum   22.201  21.312  29.931

── Multi-batch throughput ──
 batch_size  total_ms_per_spec  throughput_spec_s  enc_ms_per_spec  nar_ms_per_spec  p50_ms  p95_ms
          1             22.201             45.612            7.887           12.669  21.312  29.931
          8              3.176            321.119            1.066            1.707   2.971   4.445
         32              1.894            528.897            0.581            1.054   1.885   2.041
        128              1.997  

In [11]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — torch.profiler: NAR (bs=1, warmup=20, active=50)
# ═══════════════════════════════════════════════════════════════════════
ACTS          = ([ProfilerActivity.CPU, ProfilerActivity.CUDA]
                 if DEVICE == 'cuda' else [ProfilerActivity.CPU])
SORT_KEY      = 'cpu_time_total'
N_PROF_BATCHES = PROF_WARMUP + PROF_ACTIVE   # 70

# ── Pre-fetch 70 batches into GPU memory ─────────────────────────────
print(f'Pre-fetching {N_PROF_BATCHES} bs=1 batches for profiler…')
_dm_prof = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _skipped = [], 0
for _b in _dm_prof.predict_dataloader():
    _mz, _it, _pr, _ = model._process_batch(_b)
    if _pr[0, 1].item() > MODEL_MAX_CHARGE:
        _skipped += 1; continue
    _prof_batches.append((_mz.to(DEVICE), _it.to(DEVICE), _pr.to(DEVICE)))
    if len(_prof_batches) >= N_PROF_BATCHES:
        break

if _skipped:
    print(f'  Skipped {_skipped} out-of-range-charge spectra')
if len(_prof_batches) == 0:
    raise RuntimeError('No valid batches for profiling.')
while len(_prof_batches) < N_PROF_BATCHES:
    _prof_batches.extend(_prof_batches[:N_PROF_BATCHES - len(_prof_batches)])
print(f'Using {len(_prof_batches)} batches  '
      f'(sample charges: {[int(b[2][0,1].item()) for b in _prof_batches[:8]]}…)')

_zero_toks = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

# Quick warm-up
with torch.no_grad():
    for _mz, _it, _pr in _prof_batches[:5]:
        _me, _mk = model.encoder(_mz, _it)
        model.decoder(tokens=_zero_toks, memory=_me,
                      memory_key_padding_mask=_mk, precursors=_pr)
_sync()

# ════════════════════════════════════════════════════════════════════
# Profiler A — SpectrumEncoder
# ════════════════════════════════════════════════════════════════════
print('\n── torch.profiler  A: SpectrumEncoder (NAR, bs=1) ──')
_store_enc = {}

def _on_ready_enc(p):
    p.export_chrome_trace('results/trace_nar_encoder.json')
    _store_enc['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=10)
    _store_enc['avgs'] = p.key_averages()

with profile(
    activities=ACTS,
    record_shapes=True,
    schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
    on_trace_ready=_on_ready_enc,
) as p_enc:
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches:
            with record_function('encoder_nar'):
                model.encoder(_mz, _it)
            _sync()
            p_enc.step()

print(_store_enc.get('tbl', '(no profiler data)'))
with open('results/profiler_nar_encoder.txt', 'w') as fh:
    fh.write(f'NAR SpectrumEncoder  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
    fh.write('=' * 64 + '\n')
    fh.write(str(_store_enc.get('tbl', 'no data')))
print('Chrome trace → results/trace_nar_encoder.json')
_sync()
if DEVICE == 'cuda': torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════
# Profiler B — Full NAR forward (encoder + single decoder pass)
# ════════════════════════════════════════════════════════════════════
print('\n── torch.profiler  B: Full NAR forward (bs=1) ──')
_store_nar = {}

def _on_ready_nar(p):
    p.export_chrome_trace('results/trace_nar_full.json')
    _store_nar['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=12)
    _store_nar['avgs'] = p.key_averages()

with profile(
    activities=ACTS,
    record_shapes=True,
    schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
    on_trace_ready=_on_ready_nar,
) as p_nar:
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches:
            with record_function('nar_full_forward'):
                _me, _mk = model.encoder(_mz, _it)
                model.decoder(tokens=_zero_toks, memory=_me,
                              memory_key_padding_mask=_mk, precursors=_pr)
            _sync()
            p_nar.step()

print(_store_nar.get('tbl', '(no profiler data)'))
with open('results/profiler_nar_full.txt', 'w') as fh:
    fh.write(f'NAR full forward  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
    fh.write('=' * 64 + '\n')
    fh.write(str(_store_nar.get('tbl', 'no data')))
print('Chrome trace → results/trace_nar_full.json')

# ── CUDA kernel analysis ──────────────────────────────────────────────
# FIX: PyTorch 2.x removed cuda_time_total from FunctionEventAvg.
#      Use self_cuda_time_total (µs) which maps to the "Self CUDA" column.
#      cudaLaunchKernel itself has Self CUDA = 0 (CPU-side call), so count
#      it separately via its e.count for accurate launch-per-spectrum stats.
if DEVICE == 'cuda' and _store_nar.get('avgs'):

    # Build sorted ops list using self_cuda_time_total (µs → ms)
    def _cuda_ms(e):
        return getattr(e, 'self_cuda_time_total', 0) / 1000

    _ops = sorted(
        [(e.key, _cuda_ms(e), e.count)
         for e in _store_nar['avgs']
         if _cuda_ms(e) > 0],
        key=lambda x: -x[1])

    _cuda_total_ms = sum(ms for _, ms, _ in _ops) or 1.0

    # cudaLaunchKernel count = actual GPU kernel dispatches
    # (its Self CUDA is 0 so it won't appear in _ops above)
    _launch_entry = next(
        (e for e in _store_nar['avgs'] if e.key == 'cudaLaunchKernel'), None)
    if _launch_entry is not None:
        _total_launches = _launch_entry.count
    else:
        # Fallback: sum all op counts (over-counts but better than nothing)
        _total_launches = sum(cnt for _, _, cnt in _ops)

    nar_launches_per_spec = _total_launches / PROF_ACTIVE
    nar_cuda_total_ms     = _cuda_total_ms  / PROF_ACTIVE

    print(f'\n── CUDA kernel breakdown — Self CUDA time ({PROF_ACTIVE} spectra) ──')
    print(f'{"Kernel":<50} {"ms":>7} {"n":>7} {"%":>6}')
    print('-' * 72)
    for kn, kms, cnt in _ops[:10]:
        pct = kms / _cuda_total_ms * 100
        print(f'{kn[:50]:<50} {kms:>7.2f} {cnt:>7} {pct:>5.1f}%')

    print(f'\nTotal Self-CUDA / spectrum : {nar_cuda_total_ms:.2f} ms')
    print(f'NAR kernel launches / spectrum : {nar_launches_per_spec:.0f}')
    print(f'AR  kernel launches / spectrum : ~13 539  (Phase 1 baseline)')
    _reduction = 13539 / max(nar_launches_per_spec, 1)
    print(f'Launch reduction factor        : {_reduction:.1f}×')

else:
    nar_launches_per_spec = 0
    nar_cuda_total_ms     = 0
    print('CUDA kernel analysis: open trace at https://ui.perfetto.dev')

_sync()
if DEVICE == 'cuda': torch.cuda.empty_cache()

Pre-fetching 70 bs=1 batches for profiler…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches  (sample charges: [3, 4, 4, 3, 4, 2, 4, 2]…)

── torch.profiler  A: SpectrumEncoder (NAR, bs=1) ──
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.56%       4.417ms       100.00%     791.320ms      15.826ms       0.000us         0.00%      56.872ms       1.137ms            50  
                                            encoder_nar        13.35%     105.629ms        

In [12]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Synthetic Micro-benchmark (isolated, 20 reps, bs=1)
# Measures: encoder alone | NAR decoder alone | full NAR forward
# Compares directly with AR Phase 1 synthetic results (hard-coded).
# ═══════════════════════════════════════════════════════════════════════

# AR Phase 1 synthetic baselines
AR_SYNTH_ENC_MS   = 8.86
AR_SYNTH_FULL_MS  = 269.32
AR_SYNTH_DEC_MS   = AR_SYNTH_FULL_MS - AR_SYNTH_ENC_MS   # 260.46 ms (~18 steps)
AR_SYNTH_STEP_MS  = 14.61
AR_SYNTH_STEPS    = round(AR_SYNTH_DEC_MS / AR_SYNTH_STEP_MS)

def make_synth_batch(bs=1, device=DEVICE):
    n_real = 123
    mzs_s  = torch.zeros(bs, N_PEAKS, device=device)
    ints_s = torch.zeros(bs, N_PEAKS, device=device)
    for i in range(bs):
        mzs_s[i, :n_real]  = torch.rand(n_real, device=device) * 1303 + 301
        ints_s[i, :n_real] = torch.rand(n_real, device=device)
        norm = ints_s[i, :n_real].norm().clamp(min=1e-8)
        ints_s[i, :n_real] /= norm
    charge = 2.0; pmz = 600.0
    precs  = torch.tensor(
        [[(pmz - 1.007276) * charge, charge, pmz]] * bs,
        dtype=torch.float, device=device)
    return mzs_s, ints_s, precs

smzs, sints, sprecs = make_synth_batch(bs=1)
szero = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

# ── Warm-up ──────────────────────────────────────────────────────────
with torch.no_grad():
    for _ in range(5):
        _me, _mk = model.encoder(smzs, sints)
        model.decoder(tokens=szero, memory=_me,
                      memory_key_padding_mask=_mk, precursors=sprecs)
_sync()

# Pre-compute encoder output for isolated decoder timing
with torch.no_grad():
    smems, smasks = model.encoder(smzs, sints)
_sync()

# ── (A) Encoder — 20 reps ────────────────────────────────────────────
_et = []
with torch.no_grad():
    for _ in range(20):
        _sync(); t0 = time.perf_counter()
        model.encoder(smzs, sints)
        _sync(); _et.append((time.perf_counter() - t0) * 1000)

# ── (B) NAR Decoder — 20 reps (encoder output pre-computed) ─────────
_dt = []
with torch.no_grad():
    for _ in range(20):
        _sync(); t0 = time.perf_counter()
        model.decoder(tokens=szero, memory=smems,
                      memory_key_padding_mask=smasks, precursors=sprecs)
        _sync(); _dt.append((time.perf_counter() - t0) * 1000)

# ── (C) Full NAR forward (enc + dec) — 20 reps ───────────────────────
_ft = []
with torch.no_grad():
    for _ in range(20):
        _sync(); t0 = time.perf_counter()
        _me, _mk = model.encoder(smzs, sints)
        model.decoder(tokens=szero, memory=_me,
                      memory_key_padding_mask=_mk, precursors=sprecs)
        _sync(); _ft.append((time.perf_counter() - t0) * 1000)

# Discard first 5 reps (JIT / cache effects)
nar_enc_ms  = float(np.mean(_et[5:]))
nar_dec_ms  = float(np.mean(_dt[5:]))
nar_full_ms = float(np.mean(_ft[5:]))
nar_synth_tp = 1000.0 / nar_full_ms

print(f'\n── NAR Synthetic Micro-timings (20 reps, drop first 5) ──')
print(f'  SpectrumEncoder  : {nar_enc_ms:.2f} ms')
print(f'  NAR Decoder      : {nar_dec_ms:.2f} ms  '
      f'[{model.max_peptide_len} positions in parallel, 1 pass]')
print(f'  Full forward     : {nar_full_ms:.2f} ms  →  {nar_synth_tp:.1f} spec/s')

print(f'\n── AR vs NAR (synthetic, bs=1) ──')
_hdr = f'{"Metric":<38} {"AR":>10} {"NAR":>10} {"Speedup":>9}'
print(_hdr); print('-' * len(_hdr))
rows = [
    ('SpectrumEncoder (ms)',    AR_SYNTH_ENC_MS,  nar_enc_ms),
    ('Decoder (ms)',            AR_SYNTH_DEC_MS,  nar_dec_ms),
    ('Full forward (ms)',       AR_SYNTH_FULL_MS, nar_full_ms),
    ('Throughput (spec/s)',     1000/AR_SYNTH_FULL_MS, nar_synth_tp),
    (f'Decoder steps (AR={AR_SYNTH_STEPS}→NAR=1)', AR_SYNTH_STEPS, 1),
]
for label, ar_val, nar_val in rows:
    spd = ar_val / max(nar_val, 0.001) if label != f'Decoder steps (AR={AR_SYNTH_STEPS}→NAR=1)' else AR_SYNTH_STEPS
    print(f'  {label:<36} {ar_val:>10.2f} {nar_val:>10.2f} {spd:>8.2f}×')

pd.DataFrame({
    'metric'  : ['enc_ms', 'dec_ms', 'full_ms', 'throughput_spec_s', 'decoder_steps'],
    'AR'      : [AR_SYNTH_ENC_MS, AR_SYNTH_DEC_MS, AR_SYNTH_FULL_MS,
                 1000/AR_SYNTH_FULL_MS, AR_SYNTH_STEPS],
    'NAR'     : [nar_enc_ms, nar_dec_ms, nar_full_ms, nar_synth_tp, 1],
    'speedup' : [AR_SYNTH_ENC_MS/max(nar_enc_ms, 0.001),
                 AR_SYNTH_DEC_MS/max(nar_dec_ms, 0.001),
                 AR_SYNTH_FULL_MS/max(nar_full_ms, 0.001),
                 nar_synth_tp/(1000/AR_SYNTH_FULL_MS), AR_SYNTH_STEPS],
}).to_csv('results/nar_vs_ar_synthetic.csv', index=False)
print('\nSaved: results/nar_vs_ar_synthetic.csv')


── NAR Synthetic Micro-timings (20 reps, drop first 5) ──
  SpectrumEncoder  : 9.16 ms
  NAR Decoder      : 15.00 ms  [100 positions in parallel, 1 pass]
  Full forward     : 23.76 ms  →  42.1 spec/s

── AR vs NAR (synthetic, bs=1) ──
Metric                                         AR        NAR   Speedup
----------------------------------------------------------------------
  SpectrumEncoder (ms)                       8.86       9.16     0.97×
  Decoder (ms)                             260.46      15.00    17.37×
  Full forward (ms)                        269.32      23.76    11.33×
  Throughput (spec/s)                        3.71      42.08     0.09×
  Decoder steps (AR=18→NAR=1)               18.00       1.00    18.00×

Saved: results/nar_vs_ar_synthetic.csv


In [13]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Plots + Comprehensive Summary
# Requires: timing_summary (Cell 6), nar_*_ms (Cell 8),
#           nar_launches_per_spec (Cell 7), df_stage, df_throughput
# ═══════════════════════════════════════════════════════════════════════
import datetime

# AR baselines (real spectra, Phase 1)
AR_REAL_TOTAL_MS    = 377.46
AR_REAL_ENC_MS      = 8.15
AR_REAL_DEC_MS      = 367.15
AR_REAL_TP          = 2.6
AR_SYNTH_FULL_MS_P1 = 269.32
AR_SYNTH_ENC_MS_P1  = 8.86

# Safe read of nar_launches_per_spec (defined in Cell 7 only if CUDA)
try:
    _nlps = nar_launches_per_spec
except NameError:
    _nlps = 0

bs1 = timing_summary[1]
_lats = [timing_summary[bs]['total_mean'] for bs in BATCH_SIZES]
_tps  = [timing_summary[bs]['throughput'] for bs in BATCH_SIZES]
_encs = [timing_summary[bs]['enc_mean']   for bs in BATCH_SIZES]
_nars = [timing_summary[bs]['nar_mean']   for bs in BATCH_SIZES]
_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]

# ════════════════════════════════════════════════════════════════════
# Figure 1 — Stage breakdown: NAR vs AR side-by-side (bs=1)
# ════════════════════════════════════════════════════════════════════
fig1, ax1 = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle('Stage Breakdown (bs=1) — NAR vs AR Baseline', fontweight='bold')

_stages     = ['Fetch', 'H2D', 'Encoder', 'Decoder', 'Write']
_nar_vals   = [bs1['fetch_mean'], bs1['h2d_mean'],
               bs1['enc_mean'],   bs1['nar_mean'], bs1['write_mean']]
_ar_vals    = [1.91, 0.24, AR_REAL_ENC_MS, AR_REAL_DEC_MS, 0.01]
_nar_colors = ['#888', '#E67E22', '#378ADD', '#1D9E75', '#888']
_ar_colors  = ['#888', '#E67E22', '#378ADD', '#D85A30', '#888']

for ax, vals, colors, title in [
    (ax1[0], _nar_vals, _nar_colors,
     f'NAR — 1 parallel pass\nTotal: {bs1["total_mean"]:.1f} ms/spec'),
    (ax1[1], _ar_vals,  _ar_colors,
     f'AR — beam search (~{AR_SYNTH_STEPS} sequential steps)\nTotal: {AR_REAL_TOTAL_MS:.1f} ms/spec'),
]:
    bars = ax.bar(_stages, vals, color=colors, edgecolor='none', width=0.55)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2,
                b.get_height() + max(vals) * 0.01,
                f'{v:.1f}', ha='center', fontsize=9)
    ax.set_title(title); ax.set_ylabel('ms / spectrum')
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_stage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_stage_comparison.png')

# ════════════════════════════════════════════════════════════════════
# Figure 2 — Throughput & Latency vs Batch Size
# ════════════════════════════════════════════════════════════════════
fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('NAR Performance vs Batch Size', fontweight='bold')

# Left: Throughput
ax2[0].plot(_xi, _tps, 'o-', color='#1D9E75', lw=2, ms=8, label='NAR')
ax2[0].axhline(AR_REAL_TP, color='#D85A30', lw=1.5, ls='--',
               label=f'AR baseline ({AR_REAL_TP} spec/s)')
ax2[0].axhline(20, color='#9B59B6', lw=1.5, ls='-.',
               label='20 spec/s  (50 ms budget)')
for i, y in zip(_xi, _tps):
    ax2[0].text(i, y + max(_tps) * 0.02, f'{y:.0f}', ha='center', fontsize=9)
ax2[0].set_xticks(_xi); ax2[0].set_xticklabels(_xlbl)
ax2[0].set_xlabel('Batch size'); ax2[0].set_ylabel('Throughput (spec/s)')
ax2[0].set_title('Throughput vs Batch Size')
ax2[0].legend(fontsize=8, frameon=False)
ax2[0].spines[['top', 'right']].set_visible(False)

# Right: Stacked latency
_other = [max(0, _lats[i] - _encs[i] - _nars[i]) for i in range(len(BATCH_SIZES))]
ax2[1].bar(_xi, _encs, color='#378ADD', label='Encoder', width=0.5)
ax2[1].bar(_xi, _nars, bottom=_encs, color='#1D9E75', label='NAR Decoder', width=0.5)
ax2[1].bar(_xi, _other,
           bottom=[_encs[i] + _nars[i] for i in range(len(BATCH_SIZES))],
           color='#E67E22', label='Fetch+H2D+Write', width=0.5)
ax2[1].axhline(35, color='#9B59B6', lw=2, ls='--', label='35 ms target')
ax2[1].axhline(50, color='#E67E22', lw=1.5, ls='-.', label='50 ms ok')
ax2[1].axhline(AR_REAL_TOTAL_MS, color='#D85A30', lw=1, ls=':',
               label=f'AR total ({AR_REAL_TOTAL_MS:.0f} ms)')
for i, y in zip(_xi, _lats):
    ax2[1].text(i, y + 1, f'{y:.0f}', ha='center', fontsize=9, fontweight='bold')
ax2[1].set_xticks(_xi); ax2[1].set_xticklabels(_xlbl)
ax2[1].set_xlabel('Batch size'); ax2[1].set_ylabel('ms / spectrum')
ax2[1].set_title('Latency per Spectrum vs Batch Size')
ax2[1].legend(fontsize=8, frameon=False)
ax2[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_performance_vs_bs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_performance_vs_bs.png')

# ════════════════════════════════════════════════════════════════════
# Figure 3 — AR vs NAR synthetic comparison + Latency distribution
# ════════════════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle('AR vs NAR — Synthetic Timing + Real Latency Distribution', fontweight='bold')

# Left: bar comparison
_spd = AR_SYNTH_FULL_MS_P1 / max(nar_full_ms, 0.001)
bars3 = ax3[0].bar(['AR\n(Phase 1)', 'NAR\n(measured)'],
                    [AR_SYNTH_FULL_MS_P1, nar_full_ms],
                    color=['#D85A30', '#1D9E75'], edgecolor='none', width=0.35)
for b, v, tp in zip(bars3,
                     [AR_SYNTH_FULL_MS_P1, nar_full_ms],
                     [1000/AR_SYNTH_FULL_MS_P1, 1000/max(nar_full_ms, 0.001)]):
    ax3[0].text(b.get_x() + b.get_width() / 2,
                b.get_height() + 4,
                f'{v:.0f} ms\n{tp:.1f} spec/s', ha='center', fontsize=10)
ax3[0].axhline(35, color='#9B59B6', lw=2, ls='--', label='35 ms target')
ax3[0].axhline(50, color='#E67E22', lw=1.5, ls='-.', label='50 ms ok')
ax3[0].set_title(f'Full Forward Pass (synthetic, bs=1)\nSpeedup: {_spd:.1f}×')
ax3[0].set_ylabel('ms / spectrum')
ax3[0].legend(fontsize=9, frameon=False)
ax3[0].spines[['top', 'right']].set_visible(False)

# Right: real latency histogram (bs=1)
_raw1   = timing_summary[1]['raw']['total']
_mean_t = np.mean(_raw1)
_p50_t  = np.percentile(_raw1, 50)
_p95_t  = np.percentile(_raw1, 95)
ax3[1].hist(_raw1, bins=25, color='#1D9E75', edgecolor='white', alpha=0.85)
ax3[1].axvline(_mean_t, color='#D85A30', lw=2, ls='--', label=f'mean {_mean_t:.1f} ms')
ax3[1].axvline(_p50_t,  color='#378ADD', lw=2, ls='-.', label=f'p50  {_p50_t:.1f} ms')
ax3[1].axvline(_p95_t,  color='#9B59B6', lw=1.5, ls=':', label=f'p95  {_p95_t:.1f} ms')
ax3[1].axvline(35, color='black', lw=1.5, ls='--', alpha=0.4, label='35 ms target')
ax3[1].axvline(50, color='gray',  lw=1,   ls='--', alpha=0.4, label='50 ms ok')
ax3[1].legend(fontsize=8, frameon=False)
ax3[1].set_title(f'NAR Latency Distribution (bs=1, {len(_raw1)} spectra)')
ax3[1].set_xlabel('ms / spectrum')
ax3[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/nar_ar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_ar_comparison.png')

# ════════════════════════════════════════════════════════════════════
# Text Summary
# ════════════════════════════════════════════════════════════════════
_now    = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
_t35    = 'MEETS ✓' if bs1['total_mean'] <= 35 else f'FAILS ({bs1["total_mean"]:.1f} ms, {bs1["total_mean"]/35:.1f}× over)'
_t50    = 'MEETS ✓' if bs1['total_mean'] <= 50 else f'FAILS ({bs1["total_mean"]:.1f} ms, {bs1["total_mean"]/50:.1f}× over)'
_spd_r  = AR_REAL_TOTAL_MS  / max(bs1['total_mean'], 0.001)
_spd_s  = AR_SYNTH_FULL_MS_P1 / max(nar_full_ms, 0.001)
_lred   = f'{13539 / max(_nlps, 1):.1f}×' if _nlps > 0 else 'see trace'

summary = f"""CASANOVO NAR PROFILING — PR #548 (Non-Autoregressive Decoder)
Generated  : {_now}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
Dataset    : {SUBSET_MGF} ({N_SUBSET} spectra) | Timed: {N_TIMING_SPECTRA} spectra per batch size
Batch sizes: {BATCH_SIZES}

KEY ARCHITECTURAL CHANGE (PR #548)
  AR  → beam_search_decode: ~{AR_SYNTH_STEPS} sequential decoder passes, causal attention
  NAR → _forward_step:      1 parallel pass over {model.max_peptide_len} positions, full attention
  Mechanism: all-False tgt_mask in PeptideDecoder.embed + zero tokens to decoder

REAL-SPECTRA STAGE BREAKDOWN (bs=1, {bs1['n_spec']} spectra)
{df_stage.to_string(index=False)}
  Throughput : {bs1['throughput']:.1f} spec/s
  GPU util   : {gpu_util_mean:.0f}%  |  VRAM: {gpu_vram_peak:.2f} / {TOTAL_VRAM:.1f} GB
  35 ms target: {_t35}
  50 ms target: {_t50}

MULTI-BATCH THROUGHPUT
{df_throughput.to_string(index=False)}

SYNTHETIC MICRO-BENCHMARK (20 reps, bs=1)
  SpectrumEncoder        : {nar_enc_ms:.2f} ms  (AR: {AR_SYNTH_ENC_MS_P1:.2f} ms)
  Decoder (NAR 1 pass)   : {nar_dec_ms:.2f} ms  (AR {AR_SYNTH_STEPS} steps: {AR_SYNTH_DEC_MS:.2f} ms)
  Full forward (NAR)     : {nar_full_ms:.2f} ms  → {nar_synth_tp:.1f} spec/s
  Full forward (AR)      : {AR_SYNTH_FULL_MS_P1:.2f} ms → {1000/AR_SYNTH_FULL_MS_P1:.1f} spec/s
  Speedup (synthetic)    : {_spd_s:.1f}×
  Speedup (real spectra) : {_spd_r:.1f}×

CUDA KERNEL ANALYSIS (bs=1, {PROF_ACTIVE} profiled spectra)
  NAR launches / spectrum : {_nlps:.0f}
  AR  launches / spectrum : ~13 539  (Phase 1)
  Launch reduction factor : {_lred}

BOTTLENECK ANALYSIS (NAR, bs=1)
  Encoder  ({bs1['enc_mean']:.1f} ms, {bs1['enc_mean']/bs1['total_mean']*100:.0f}%): identical to AR; irreducible without BF16/compile
  Decoder  ({bs1['nar_mean']:.1f} ms, {bs1['nar_mean']/bs1['total_mean']*100:.0f}%): down from {AR_REAL_DEC_MS:.0f} ms — dominant speedup
  Batch scaling: tp at bs={BATCH_SIZES[-1]}: {timing_summary[BATCH_SIZES[-1]]['throughput']:.0f} spec/s vs {bs1['throughput']:.0f} at bs=1

ARTIFACTS SAVED TO results/
  nar_stage_comparison.png       stage bars NAR vs AR
  nar_performance_vs_bs.png      throughput + latency vs batch size
  nar_ar_comparison.png          synthetic benchmark + latency histogram
  trace_nar_encoder.json         encoder Chrome trace  (ui.perfetto.dev)
  trace_nar_full.json            full NAR forward trace
  profiler_nar_encoder.txt       key_averages encoder
  profiler_nar_full.txt          key_averages full forward
  nar_stage_timing_bs1.csv
  nar_throughput_all_bs.csv
  nar_vs_ar_synthetic.csv
  nar_summary.txt                this summary
  nar_benchmark_table.csv        machine-readable KPIs
"""

print(summary)
with open('results/nar_summary.txt', 'w') as fh:
    fh.write(summary)

pd.DataFrame({
    'metric': ['nar_bs1_total_ms', 'nar_bs1_tp', 'ar_bs1_total_ms', 'ar_bs1_tp',
               'speedup_real', 'speedup_synth',
               'kernel_launches_nar', 'kernel_launches_ar'],
    'value' : [round(bs1['total_mean'], 2), round(bs1['throughput'], 1),
               AR_REAL_TOTAL_MS, AR_REAL_TP,
               round(_spd_r, 2), round(_spd_s, 2),
               round(_nlps, 0), 13539],
}).to_csv('results/nar_benchmark_table.csv', index=False)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<55} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nNAR profiling complete. Primary: results/nar_summary.txt')

Saved: results/nar_stage_comparison.png
Saved: results/nar_performance_vs_bs.png
Saved: results/nar_ar_comparison.png
CASANOVO NAR PROFILING — PR #548 (Non-Autoregressive Decoder)
Generated  : 2026-06-14 23:53
Hardware   : NVIDIA L4 | 23.6 GB VRAM | PyTorch 2.7.1+cu128
Dataset    : subset_profile.mgf (6000 spectra) | Timed: 5000 spectra per batch size
Batch sizes: [1, 8, 32, 128, 512]

KEY ARCHITECTURAL CHANGE (PR #548)
  AR  → beam_search_decode: ~18 sequential decoder passes, causal attention
  NAR → _forward_step:      1 parallel pass over 100 positions, full attention
  Mechanism: all-False tgt_mask in PeptideDecoder.embed + zero tokens to decoder

REAL-SPECTRA STAGE BREAKDOWN (bs=1, 5000 spectra)
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.334   1.276   1.845
        H2D transfer    0.210   0.197   0.295
     SpectrumEncoder    7.887   7.480  11.657
NAR Decoder (1 pass)   12.669  12.174  16.541
        Output write    0.103   0.094   0.132
  TOTAL per s